# Grover search for complementary DNA bases

This minimal example searches the 16 ordered pairs of two-bit DNA bases and marks the four complementary pairs. Because $N=16$ and $M=4$, one Grover iteration raises the ideal probability of the valid subspace from $1/4$ to $1$.

## Register and ordering

The four search qubits are `q0, q1, q2, q3`. In the wire-order string $b_0b_1b_2b_3$, `(q0, q1)` encodes the first base and `(q2, q3)` encodes the second. The encoding is `A=00`, `C=01`, `G=10`, and `T=11`.

OpenQARP labels basis states least-significant-bit first: qubit 0 is the least-significant bit of the integer label. `Grover` measures the complete search register in qubit order, so a sampler key is `(b0, b1, b2, b3)` and this notebook displays it by joining the tuple directly. Thus the displayed DNA bit string is a **wire-order string**, not the usual most-significant-bit-first rendering of an integer. The four wire-order strings `0011`, `1100`, `0110`, and `1001` have OpenQARP integer labels 12, 3, 6, and 9, respectively.

In [ ]:
import itertools
from copy import deepcopy

import numpy as np

from qarp.algorithms import Grover, Sampler
from qarp.blocks import SimpleBlock
from qarp.endianness import bits_to_label
from qarp.engines import QarpEngine

BASE_FROM_BITS = {(0, 0): "A", (0, 1): "C", (1, 0): "G", (1, 1): "T"}
COMPLEMENT = {"A": "T", "T": "A", "C": "G", "G": "C"}
EXPECTED = {
    "0011": "A-T",
    "1100": "T-A",
    "0110": "C-G",
    "1001": "G-C",
}


def wire_bits(bit_string):
    return tuple(int(bit) for bit in bit_string)


def decode(bits):
    return f"{BASE_FROM_BITS[bits[:2]]}-{BASE_FROM_BITS[bits[2:]]}"


valid_bits = {wire_bits(bit_string) for bit_string in EXPECTED}
# good_states uses OpenQARP's LSB-first integer labels; it is reporting metadata only.
good_states = [bits_to_label(bits) for bits in valid_bits]
assert set(good_states) == {3, 6, 9, 12}

## Reversible complementarity oracle

The current `Grover` API treats every oracle qubit as a search qubit: it prepares and diffuses across the oracle's entire width. Consequently, adding separate workspace qubits would change this from a four-qubit search. No extra ancillas are needed here because the two condition values can be computed reversibly **in place**:

1. `CX(q0, q2)` temporarily changes `q2` to $b_0\oplus b_2$.
2. `CX(q1, q3)` temporarily changes `q3` to $b_1\oplus b_3$.
3. `CZ(q2, q3)` applies a phase of $-1$ exactly when both temporary condition bits are 1; this is their logical AND.
4. The two CNOTs are repeated in reverse order, restoring `q2` and `q3` to their original base bits.

The resulting four-qubit block is the required phase oracle $I-2\Pi_{\mathrm{complementary}}$. The temporary condition values are fully uncomputed before the oracle returns.

In [ ]:
oracle = SimpleBlock(4, name="DNA_complementarity")
oracle.cx(0, 2)  # Compute b0 XOR b2 in q2.
oracle.cx(1, 3)  # Compute b1 XOR b3 in q3.
oracle.cz(2, 3)  # Phase flip iff both complementarity conditions are true.
oracle.cx(1, 3)  # Uncompute q3.
oracle.cx(0, 2)  # Uncompute q2.

assert oracle.n_qubits == 4

# Independent check: enumerate every basis state, decode it, and keep the ones
# the base-pairing rule calls complementary — then require the oracle unitary to
# be exactly I - 2*Pi over that set.  Exact, not modulo phase: the oracle's
# global phase becomes relative inside Grover's controlled diffusion.
marked = {
    bits_to_label(bits)
    for bits in itertools.product((0, 1), repeat=4)
    if BASE_FROM_BITS[bits[2:]] == COMPLEMENT[BASE_FROM_BITS[bits[:2]]]
}
assert marked == set(good_states)

reference = np.eye(2**4, dtype=complex)
for label in marked:
    reference[label, label] = -1.0
assert np.array_equal(deepcopy(oracle).build().unitary_matrix(), reference)

## One Grover iteration and four-qubit measurement

`Grover` supplies the uniform Hadamard preparation and its existing diffusion operator. Passing `n_marked=4` makes the current known-count API select one iteration. The sampler uses 20,000 shots, and the seeded ideal OpenQARP engine makes this executable example reproducible. Only the four search qubits are measured because the oracle itself is four qubits wide.

In [ ]:
search = Grover(
    oracle,
    n_marked=4,
    good_states=good_states,
    primitive=Sampler(n_shots=20_000),
    engine=QarpEngine(seed=7),
).build()
distribution = search.run()

assert search.n_iterations == 1
assert search.primitive.measured_qubits == [0, 1, 2, 3]

## Decode and check the result

Ideally, `0011` (A-T), `1100` (T-A), `0110` (C-G), and `1001` (G-C) each occur with probability $1/4$, while every invalid pair has probability zero. Finite-shot frequencies fluctuate around $0.25$, so the check allows three percentage points per valid state and at most one percent total invalid probability.

In [ ]:
for bit_string, expected_pair in EXPECTED.items():
    bits = wire_bits(bit_string)
    probability = distribution.get(bits, 0.0)
    assert decode(bits) == expected_pair
    assert abs(probability - 0.25) < 0.03
    print(f"{bit_string}  {decode(bits):>3}  {probability:.4f}")

invalid_probability = sum(
    probability for bits, probability in distribution.items() if bits not in valid_bits
)
assert invalid_probability < 0.01
assert search.success_probability is not None and search.success_probability > 0.99

print(f"Invalid-pair probability: {invalid_probability:.4f}")
print(f"Total valid-pair probability: {search.success_probability:.4f}")

## Plot the amplified distribution

The histogram is restricted to the four valid wire-order strings. The automated check above independently guards against hiding appreciable probability on an invalid state.

In [ ]:
from qarp.plotting import plot_histogram

valid_distribution = {
    wire_bits(bit_string): distribution.get(wire_bits(bit_string), 0.0) for bit_string in EXPECTED
}
assert len(valid_distribution) == 4

figure, axis = plot_histogram(
    valid_distribution,
    title="Grover search: complementary DNA base pairs",
    figsize=(7, 4),
    return_plotter=True,
)
axis.set_xticks(
    range(len(EXPECTED)),
    [f"{bit_string}\n{pair}" for bit_string, pair in EXPECTED.items()],
    rotation=0,
)
axis.set_xlabel("Valid state (wire-order bits and DNA pair)")
figure

## Plot the complete Grover circuit

This expands OpenQARP's composite boxes so the diagram shows the uniform Hadamard preparation, the compute-phase-uncompute DNA oracle, and the diffusion operations used in the single Grover iteration.

In [ ]:
assert search.block is not None
search.block.plot(
    decompose_boxes=True,
    scrollable=False,
    interactive=False,
    figsize=(16, 3.2),
)

## Extension: amplify allowed four-base chains

Use eight search qubits for four bases, with `(q0, q1)`, `(q2, q3)`, `(q4, q5)`, and `(q6, q7)` holding the successive base encodings. A chain is allowed when bases 0 and 1 are complementary **and** bases 2 and 3 are complementary. For example, `ATCG` is allowed because it consists of the valid pairs `A-T` and `C-G`.

There are $N=4^4=256$ four-base chains and $M=4\times4=16$ allowed chains. `build_complementarity_oracle(n_bases)` below generates the complete compute-phase-uncompute oracle for any positive even number of bases. For four bases it computes the four bitwise conditions on `q2`, `q3`, `q6`, and `q7`; for six bases it automatically adds the next paired link on `q8` through `q11`. Changing `n_bases = 4` to `n_bases = 6` is therefore enough to build the larger oracle.

In [ ]:
BITS_FROM_BASE = {base: bits for bits, base in BASE_FROM_BITS.items()}


def number_of_base_pairs(n_bases):
    if isinstance(n_bases, bool) or not isinstance(n_bases, int):
        raise TypeError("n_bases must be an integer")
    if n_bases < 2 or n_bases % 2:
        raise ValueError("n_bases must be a positive even number")
    return n_bases // 2


def build_complementarity_oracle(n_bases):
    """Mark chains made of successive complementary base pairs."""
    n_pairs = number_of_base_pairs(n_bases)
    oracle = SimpleBlock(2 * n_bases, name=f"{n_bases}_base_pairing")
    cnot_pairs = []
    condition_qubits = []

    for pair_index in range(n_pairs):
        first_base_start = 4 * pair_index
        second_base_start = first_base_start + 2
        for bit_offset in range(2):
            control = first_base_start + bit_offset
            target = second_base_start + bit_offset
            cnot_pairs.append((control, target))
            condition_qubits.append(target)

    for control, target in cnot_pairs:
        oracle.cx(control, target)
    oracle.mcz(condition_qubits)
    for control, target in reversed(cnot_pairs):
        oracle.cx(control, target)
    return oracle


def valid_complementary_chains(n_bases):
    n_pairs = number_of_base_pairs(n_bases)
    chains = [""]
    for _ in range(n_pairs):
        chains = [prefix + base + COMPLEMENT[base] for prefix in chains for base in "ACGT"]
    return chains


def encode_sequence(sequence):
    return tuple(bit for base in sequence for bit in BITS_FROM_BASE[base])


def decode_sequence(bits):
    return "".join(BASE_FROM_BITS[bits[index : index + 2]] for index in range(0, len(bits), 2))


n_bases = 4  # Change this value to test different numbers of bases (must be even).
valid_chains = valid_complementary_chains(n_bases)
valid_chain_bits = {encode_sequence(chain) for chain in valid_chains}
valid_chain_labels = [bits_to_label(encode_sequence(chain)) for chain in valid_chains]

chain_oracle = build_complementarity_oracle(n_bases)

assert len(valid_chains) == len(valid_chain_bits) == 4 ** number_of_base_pairs(n_bases)
assert all(
    chain[index + 1] == COMPLEMENT[chain[index]]
    for chain in valid_chains
    for index in range(0, n_bases, 2)
)
assert chain_oracle.n_qubits == 2 * n_bases

### Run the eight-qubit search

After three iterations, the ideal total probability on the 16 allowed chains is about $0.9613$. Each allowed chain has the same ideal probability, about $0.0601$, while the remaining probability is spread thinly across 240 invalid chains.

In [ ]:
chain_search = Grover(
    chain_oracle,
    n_marked=len(valid_chain_labels),
    good_states=valid_chain_labels,
    primitive=Sampler(n_shots=20_000),
    engine=QarpEngine(seed=11),
).build()
chain_distribution = chain_search.run()

most_likely_bits = sorted(chain_distribution, key=chain_distribution.get, reverse=True)[
    : len(valid_chain_bits)
]
assert chain_search.primitive.measured_qubits == list(range(2 * n_bases))
assert set(most_likely_bits) == valid_chain_bits
assert chain_search.success_probability is not None
assert abs(chain_search.success_probability - chain_search.predicted_success_probability) < 0.01

print(f"Allowed chains:                 {', '.join(valid_chains)}")
print(f"Grover iterations:              {chain_search.n_iterations}")
print(f"Predicted allowed probability:  {chain_search.predicted_success_probability:.6f}")
print(f"Sampled allowed probability:    {chain_search.success_probability:.6f}")
print(f"Sampled invalid probability:    {1 - chain_search.success_probability:.6f}")

### Plot the amplified four-base configurations

The 16 dominant outcomes below are precisely the allowed chains: each adjacent two-base group is one of `AT`, `TA`, `CG`, or `GC`. The sampled invalid probability reported above is retained separately rather than hidden by the filtered plot.

In [ ]:
allowed_chain_distribution = {
    encode_sequence(chain): chain_distribution.get(encode_sequence(chain), 0.0)
    for chain in valid_chains
}
assert len(allowed_chain_distribution) == len(valid_chains)

chain_figure, chain_axis = plot_histogram(
    allowed_chain_distribution,
    title=f"Grover search: allowed {n_bases}-base chains",
    figsize=(12, 4),
    return_plotter=True,
)
chain_axis.set_xticks(range(len(valid_chains)), valid_chains, rotation=45, ha="right")
chain_axis.set_xlabel("Allowed four-base chain")
chain_figure